# Introduction (TBD)

# Libraries and Global Variables

In [ ]:
#Data Analysis and manupalation libraries
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from pathlib import Path

#Sklearn
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from scipy import stats
#Seaborn
import seaborn as sns

#Matplotlib
import matplotlib.pyplot as plt

In [ ]:
#Seed for ML Algorithms
SEED = 81

#Paths
PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
GENERATED_REPORTS = (PROJECT / "reports" / "generated")

DIABETES_CSV = PROJECT / "data" / "raw" / "diabetic_data.csv"
IDS_MAPING_CSV = PROJECT / "data" / "raw" / "IDS_mapping.csv"


## Diabetes fact table

In [ ]:
#Read diabetes CSV
df_diabetes = pd.read_csv(DIABETES_CSV, delimiter=',', na_values='?', keep_default_na=False, low_memory=False)

#CATEGORIZE VARIABLES

## NUMERIC
NUMERIC = [
    "encounter_id",
    "patient_nbr",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

CATEGORICAL = [
    "race",
    "gender",
    "age",
    "weight",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
    "readmitted"
]

In [ ]:
df_diabetes.head()

## IDS Mapping dim tables
TODO: Add small explanation behind logic of generator 

In [ ]:
def process_ids_tables():
    #Read IDS mapping CSV
    df_ids_map = pd.read_csv(IDS_MAPING_CSV, header=None, names=range(2), delimiter=',')

    #Define table names
    table_names = ['admission_type', 'discharge_disposition_id', 'admission_source_id']

    groups = df_ids_map[0].isin(table_names).cumsum()

    tables = {g.iloc[0,0]: g.iloc[0:-1] for k,g in df_ids_map.groupby(groups)}

    for name, table in tables.items():
        #Create dataframe
        df = pd.DataFrame(table)

        #Promote first line to header
        df.columns = df.iloc[0]
        df = df.iloc[1:].reset_index(drop=True)

        yield df

ids_mapping_dataframes = process_ids_tables()

#Use generator to create our dataframes for each table

#Admission Types
df_admission_type = next(ids_mapping_dataframes)

#Cast admission_type_id to int for consistency with df_diabetes
df_admission_type['admission_type_id'] = df_admission_type['admission_type_id'].astype(int)

#Discharge Disposition
df_discharge_disposition = next(ids_mapping_dataframes)

#Cast discharge_disposition_id to int for consistency with df_diabetes
df_discharge_disposition['discharge_disposition_id'] = df_discharge_disposition['discharge_disposition_id'].astype(int)

#Admission Source
df_admission_source = next(ids_mapping_dataframes)

#Cast admission_source_id to int for consistency with df_diabetes
df_admission_source['admission_source_id'] = df_admission_source['admission_source_id'].astype(int)

### Admission Type mapping

In [ ]:
print(f"Min Id: {df_admission_type['admission_type_id'].min()}")
print(f"Max Id: {df_admission_type['admission_type_id'].max()}")

n_rows = df_admission_type.shape[0]
df_admission_type[:n_rows]

### Discharge Disposition mapping

In [ ]:
print(f"Min Id: {df_discharge_disposition['discharge_disposition_id'].min()}")
print(f"Max Id: {df_discharge_disposition['discharge_disposition_id'].max()}")

n_rows = df_discharge_disposition.shape[0]
df_discharge_disposition[:n_rows]

### Admission Source mapping

In [ ]:
print(f"Min Id: {df_admission_source['admission_source_id'].min()}")
print(f"Max Id: {df_admission_source['admission_source_id'].max()}")

n_rows = df_admission_source.shape[0]
df_admission_source[:n_rows]

# EDA

Just for starters, we'll create a Profile Report for the diabetes main fact table

In [ ]:
def create_profile_report(df, title, file_name):

    profile = ProfileReport(
        df, title = title
    )
    profile.config.html.inline = True

    path_to_html = Path(PROJECT / "reports" / "generated" / file_name)

    if not path_to_html.exists():
        profile.to_file(GENERATED_REPORTS / file_name)

#Create profile report for this dataframe
create_profile_report(df_diabetes, "Diabetes Data Profile Report", "profile-report-diabetes.html")

## Administrative Data
Here we'll explore data regarding administrative data of the patient. This includes the following columns:
- admission_type_id
- admission_source_id
- discharge_disposition_id
- payer_code
- medical_specialty


### Preview, describe, nulls

In [ ]:
#Create a list with columns of administrative data
admin_data_cols = [
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "payer_code",
    "medical_specialty"
]

#Create a daframe with only these columns
df_admin_data = df_diabetes[admin_data_cols].copy()

#check dtypes
print(df_admin_data.dtypes)

df_admin_data.tail(10)

In [ ]:
#Create profile report for this dataframe
create_profile_report(df_admin_data, "Diabetes - Administrative Data Profile Report", "profile-report-admin-data.html")        

In [ ]:
#Describe numerical columns
df_admin_data.describe()

**Note to self:** Last id 29 for discharge disposition seems like it's not being used

In [ ]:
#Describe categorical columns
df_admin_data.describe(include='object')

In [ ]:
#Percentage of nan values across dataframe
round(df_admin_data.replace('?', np.nan).isna().sum() / df_admin_data.shape[0] * 100, 2)

**Key Takeaway:** Columns payer_code and medical_specialty have high null values. 39% and 49% respectively

In [ ]:
#Check distinct values of payer code for context
pd.DataFrame(df_admin_data['payer_code'].unique())

Not much to conclude from these values. Looks like country codes but verifying some of them like "OG" they do not correspond to any country.
There is no DIM table for these column. Just keep values as is (mentioned in project guide) and later decide what to do with null values. 

### Distribution

In [ ]:
#Plot distribution in bar charts
fig, axes = plt.subplots(2, 3, figsize=(12, 10))

for ax, col in zip(axes.flatten(), df_admin_data.columns):
    sns.countplot(df_admin_data, x=col, ax=ax)
    ax.set_title(f"Distribution of {col}", fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

Note: Check why the first value of these columns is almost always the value that occurs more frequently

Looking at the dimension tables displayed above we see:
- Admission Type: 1 -> Emergency
- Admission Source: 7 -> Emergency Room
- Discharge Disposition: 1 -> Discharge to home
- Payer code: ? -> Null
- Medical Specialty: ? -> Null

We can conclude that the majority of the occurences are emergency related and that the patient is discharged to home


### Correlation

In [ ]:
#This dataset has int and string values, so I'll encode the str values to int for an improved correlation

df_admin_data_encod = df_admin_data.copy()
#Columns: payer_code, medical_specialty
le = LabelEncoder()
df_admin_data_encod['payer_code'] = le.fit_transform(df_admin_data_encod['payer_code'])
df_admin_data_encod['medical_specialty'] = le.fit_transform(df_admin_data_encod['medical_specialty'])

print(df_admin_data_encod.dtypes)

df_admin_data_encod[['payer_code', 'medical_specialty']].tail(10)

In [ ]:
# Build correlation matrix
# matrix = df_admin_data_encod.corr()

# Build plot
plt.figure(figsize=(6, 5))

#Scale values using min-max scaling to improve correlation visualization
scaler = MinMaxScaler()
df_corr_analysis = pd.DataFrame(
    scaler.fit_transform(df_admin_data_encod),
    columns=df_admin_data_encod.columns,
    index=df_admin_data_encod.index,
)

# Plot heatmap using spearman correlation
sns.heatmap(
    df_admin_data_encod.corr("spearman"),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True,
    linewidth=0.5,
    vmin=-1,
    vmax=1,
)

plt.title("Correlation Analysis - Administrative Data", fontsize=16)
plt.tight_layout()
plt.show()

Let's check for correlation between columns that have similar meaning using qui-square:  
- admission_type_id <-> admission_source_id


## Utilization history/current stay data

### Preview, describe, nulls

In [ ]:
util_stay_cols = [
    "num_lab_procedures",
    "num_procedures",
    "number_diagnoses",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "time_in_hospital"
]

df_util_stay_data = df_diabetes[util_stay_cols].copy()

#dtypes
print(df_util_stay_data.dtypes)

df_util_stay_data.head(10)

In [ ]:
#Describe
df_util_stay_data.describe()

### Distribution
Check distribution of these numerical variables. The skewness of the distribution is important to determine what method of correlation should be used in the next step

In [ ]:
df_util_stay_data.columns

In [ ]:
#Statistics of data
for col in df_util_stay_data.columns:
    data = df_util_stay_data[col]

    mean_val = np.mean(data)
    std_val = np.std(data)
    iqr_val = stats.iqr(data)

    print(f"{col}: \tMean Val: {mean_val} \tStandard Deviation: {std_val} \tIQR Val: {iqr_val}")

#Plot distribution in bar charts
fig, axes = plt.subplots(2, 4, figsize=(16, 10))

#Create bins dict
bins_dict = {
    "num_lab_procedures": 15,
    "num_procedures": 20,
    "num_medications": 20,
    "number_diagnoses": 10,
    "number_outpatient": 5,
    "number_emergency": 5,
    "number_inpatient": 50,
    "time_in_hospital": 10,
}

for ax, col in zip(axes.flatten(), df_util_stay_data.columns):
    sns.histplot(df_util_stay_data, x=col, ax=ax, bins=bins_dict[col], kde=True)
    ax.set_title(f"Distribution of {col}", fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

**Conclusion:** Looking at the distribution plots (TBD)

Let's plot the distribution of number_outpatient, number_emergency and number_inpatient in plots more tightly adjusted to better view the distributions, although they are most certainly right skewed

In [ ]:
#Plot distribution in bar charts
fig, ax = plt.subplots(1, 3, figsize=(12, 6))
cols = ["number_outpatient", "number_emergency", "number_inpatient"]

for ax, col in zip(ax, cols):
    sns.histplot(df_util_stay_data, x=col, discrete=True, kde=False, ax=ax)
    ax.set_title(f"Distribution of {col}", fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

**Conclusion** We can definitely conclude that all three variables are highly right skewed and presenting a high likelihood of having outliers. This also verifiable by the values of UIQR presented above where the difference between the first and thrid quartiles are very slim, which tells us the values are concentraded around the mean

### Correlation (with outliers)

In [ ]:
# Build correlation matrix
# matrix = df_util_stay_da.corr()

# Build plot
plt.figure(figsize=(6, 5))

#Scale values using min-max scaling to improve correlation visualization
scaler = MinMaxScaler()
df_corr_analysis = pd.DataFrame(
    scaler.fit_transform(df_util_stay_data),
    columns=df_util_stay_data.columns,
    index=df_util_stay_data.index,
)

# Plot heatmap using spearman correlation
sns.heatmap(
    df_corr_analysis.corr(method='pearson'),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True,
    linewidth=0.5,
    vmin=-1,
    vmax=1,
)

plt.title("Correlation Analysis - Utilization and Current Stay Data", fontsize=16)
plt.tight_layout()
plt.show()

### Correlation (without outliers)
Remove outliers from dataframe and quickly check if correlation is more relevant

In [ ]:
#Upper trimm data in order to remove outliers
df_util_outliers = df_util_stay_data.copy()

cutoffs = {}

for c in df_util_outliers.columns:
    q1, q3 = df_util_outliers[c].quantile([0.25, 0.75])
    cutoffs[c] = df_util_outliers[c].quantile(0.90)

mask = pd.concat([df_util_outliers[c] <= cutoffs[c] for c in df_util_outliers.columns], axis=1).all(axis=1)
clean = df_util_outliers.loc[mask, df_util_outliers.columns]

corr = clean.corr(method = 'spearman')
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation matrix (upper outliers removed)")
plt.show()

**Conclusion:** Very similar correlation heatmap compared to the one with outliers, being the only difference where the correlation of medications and lab procedures dropped by about in half

Ultimately, we can conclude that, in general, these columns aren't very correlated or similar in anyway. 

There is some correlation regarding the number of medications, where the higher the lab procedures, normal procedures and time spent in hospital are, the higher the number of medication is prescribed, which makes total sense.
Also, and although not highly correlated, the higher the time spent in hospital, the more medication is prescribed.

**We should also make sure to remove outliers in pre-processing before feeding the data to our model for atleast these columns:**
- number_diagnoses
- number_outpatient
- number_emergency
- number_inpatient